# Machine Learning Fundamentals for AI Engineers

**Purpose:** Before you can deploy, optimize, or monitor ML models in production, you must understand what a model IS, how training works, and what the key metrics mean. This notebook covers the essential ML concepts referenced throughout the curriculum.

**Why this exists:** Notebook 09 (Model Development) dives into QLoRA fine-tuning and knowledge distillation. Without understanding what training, loss, and overfitting mean, those techniques are just magic code.

---

## 1 -- What IS a Machine Learning Model?

### The Simplest Explanation

A model is a **mathematical function that learns patterns from data**.

```
Traditional programming:  Rules + Data  -> Output
Machine learning:         Data + Output -> Rules (the model learns the rules!)
```

**Analogy:** Imagine teaching a child to identify cats:
- Traditional: "A cat has pointy ears, whiskers, and a tail" (you write the rules)
- ML: Show 10,000 pictures labeled "cat" and "not cat" (the model figures out the rules)

### Types of Machine Learning

| Type | What It Does | Example | Used In Curriculum |
|------|-------------|---------|--------------------|
| **Supervised** | Learns from labeled examples | Spam detection (email -> spam/not spam) | NB09, NB10 |
| **Unsupervised** | Finds patterns without labels | Customer segmentation | NB07 (embeddings) |
| **Self-supervised** | Creates its own labels from data | LLM pre-training (predict next word) | NB09 (fine-tuning) |
| **Reinforcement** | Learns by trial and error + rewards | Game AI, robot control | NB08 (Ray RLlib) |

### What Happens Inside a Model?

At its core, every model is: **output = f(input, weights)**
- **Input:** Your data (text, images, numbers)
- **Weights:** Numbers the model adjusts during training (millions to billions)
- **Output:** Prediction (class label, number, generated text)

Training = finding the best weights so the output matches reality.


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Before you throw a 70B parameter LLM at a problem, ask yourself: 'Can I solve this with Logistic Regression or a Random Forest?' Seniors know that simple, interpretative models are vastly cheaper and easier to maintain.

**Mental Model:** Supervised Learning is giving a computer a history textbook (data + answers) so it can pass a future exam (predictions). Unsupervised Learning is giving it a pile of puzzle pieces and asking it to group similar colors.

**Common Junior Pitfall:** Overfitting. Building a highly complex model that gets 99% accuracy on the training data, but violently fails on new real-world data because it 'memorized' the noise rather than learning the pattern.

---


In [ ]:
# Cell 1 -- See a model learn in real time
import numpy as np
import matplotlib.pyplot as plt

# Generate data: y = 2x + 3 + noise
np.random.seed(42)
X = np.random.uniform(0, 10, 50)
y_true = 2 * X + 3 + np.random.normal(0, 1, 50)

# Start with random weights
w = 0.0  # slope (should learn: 2)
b = 0.0  # intercept (should learn: 3)
learning_rate = 0.01

print('Watch the model LEARN the pattern:')
print(f'  True rule: y = 2x + 3')
print(f'  Starting guess: y = {w:.1f}x + {b:.1f}\n')

losses = []
for epoch in range(100):
    # Forward pass: make predictions with current weights
    y_pred = w * X + b

    # Calculate loss: how wrong are we? (Mean Squared Error)
    loss = np.mean((y_pred - y_true) ** 2)
    losses.append(loss)

    # Backward pass: which direction should weights move?
    dw = np.mean(2 * (y_pred - y_true) * X)  # gradient for slope
    db = np.mean(2 * (y_pred - y_true))       # gradient for intercept

    # Update weights (move opposite to gradient)
    w -= learning_rate * dw
    b -= learning_rate * db

    if epoch in [0, 9, 49, 99]:
        print(f'  Epoch {epoch+1:3d}: loss={loss:.4f}, learned: y = {w:.2f}x + {b:.2f}')

print(f'\nFinal learned: y = {w:.2f}x + {b:.2f}')
print(f'True formula:  y = 2.00x + 3.00')
print(f'Close enough? {abs(w-2) < 0.2 and abs(b-3) < 0.5}')


---
## 2 -- Training: The Learning Process

### Key Training Concepts

| Term | Meaning | Analogy |
|------|---------|--------|
| **Epoch** | One pass through ALL training data | Reading the entire textbook once |
| **Batch** | A small subset of data processed together | Reading one chapter |
| **Batch size** | Number of examples per batch | Number of pages per chapter |
| **Loss** | How wrong the model is (lower = better) | Exam score (inverted: 0 = perfect) |
| **Learning rate** | How big each weight adjustment is | How much you change your study method each time |
| **Gradient** | Direction to move weights to reduce loss | Compass pointing toward the answer |

### The Training Loop (What Happens in Every ML Framework)

```python
for epoch in range(num_epochs):        # Repeat many times
    for batch in training_data:         # Process data in chunks
        predictions = model(batch)      # Forward pass: predict
        loss = compare(predictions, truth)  # How wrong?
        gradients = loss.backward()     # Which direction to fix?
        optimizer.step(gradients)       # Update weights
```

This exact loop is inside HuggingFace Trainer (NB09), Ray Train (NB08), and every other training framework.

### Hyperparameters vs Parameters

| Parameters (model learns these) | Hyperparameters (YOU choose these) |
|---|---|
| Weights and biases | Learning rate |
| Attention matrices | Batch size |
| Embedding vectors | Number of epochs |
| Millions to billions of numbers | 5-20 choices |

Optuna (NB09) automates hyperparameter search so you don't guess manually.


In [ ]:
# Cell 2 -- Visualize learning rate effects

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
learning_rates = [0.001, 0.01, 0.1]
titles = ['Too small (0.001)', 'Just right (0.01)', 'Too large (0.1)']

for ax, lr, title in zip(axes, learning_rates, titles):
    w, b = 0.0, 0.0
    lr_losses = []
    for epoch in range(100):
        y_pred = w * X + b
        loss = np.mean((y_pred - y_true) ** 2)
        lr_losses.append(min(loss, 200))  # Cap for display
        dw = np.mean(2 * (y_pred - y_true) * X)
        db = np.mean(2 * (y_pred - y_true))
        w -= lr * dw
        b -= lr * db
    ax.plot(lr_losses, linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_ylim(0, 100)

plt.suptitle('Effect of Learning Rate on Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('learning_rate_comparison.png', dpi=100)
plt.show()
print('Left:   Too slow -- barely improving after 100 epochs')
print('Middle: Just right -- smooth convergence')
print('Right:  Too fast -- overshoots and may diverge')


---
## 3 -- Overfitting: The #1 Problem in ML

### What is Overfitting?

A model that MEMORIZES training data instead of LEARNING patterns.

**Analogy:** A student who memorizes every practice exam answer but can't solve new problems.

```
Training data:  [1->2, 2->4, 3->6]   Model learns: y = 2x (GOOD!)
Overfit model:  [1->2, 2->4, 3->6]   Model memorizes: {1:2, 2:4, 3:6}
                New input: 5           Good model: 10. Overfit model: ??? (never seen 5)
```

### How to Detect Overfitting

| Sign | Meaning |
|------|--------|
| Training loss keeps decreasing | Model is still learning |
| Validation loss starts INCREASING | OVERFITTING! Model memorizes train data |
| Train accuracy 99%, validation 70% | Classic overfit |

### How to Prevent Overfitting

| Method | How | Used In Curriculum |
|--------|-----|---|
| More data | Harder to memorize | NB04 (data management) |
| Regularization | Penalize large weights | NB09 (weight decay) |
| Dropout | Randomly turn off neurons | NB09 (LoRA config) |
| Early stopping | Stop when val loss increases | NB09 (trainer callbacks) |
| Cross-validation | Test on multiple data splits | NB09 (Optuna) |


In [ ]:
# Cell 3 -- Demonstrate overfitting vs good fit

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Generate data with curve + noise
X_curve = np.linspace(0, 4, 40).reshape(-1, 1)
y_curve = np.sin(X_curve).ravel() + np.random.normal(0, 0.2, 40)
X_train, X_test, y_train, y_test = train_test_split(X_curve, y_curve, test_size=0.3, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
degrees = [1, 4, 15]
titles = ['Underfit (too simple)', 'Good fit', 'Overfit (too complex)']

X_plot = np.linspace(0, 4, 200).reshape(-1, 1)
for ax, deg, title in zip(axes, degrees, titles):
    poly = PolynomialFeatures(deg)
    X_tr_poly = poly.fit_transform(X_train)
    X_te_poly = poly.transform(X_test)
    model = LinearRegression().fit(X_tr_poly, y_train)
    train_score = model.score(X_tr_poly, y_train)
    test_score = model.score(X_te_poly, y_test)

    ax.scatter(X_train, y_train, c='blue', s=30, label='Train')
    ax.scatter(X_test, y_test, c='red', s=30, label='Test')
    y_plot = model.predict(poly.transform(X_plot))
    ax.plot(X_plot, np.clip(y_plot, -3, 3), 'g-', linewidth=2)
    ax.set_title(f'{title}\nTrain: {train_score:.2f}, Test: {test_score:.2f}', fontweight='bold')
    ax.set_ylim(-2, 2)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('overfitting_demo.png', dpi=100)
plt.show()
print('Left:   Model too simple, misses the pattern (underfit)')
print('Middle: Model captures the trend without memorizing noise (ideal)')
print('Right:  Model memorizes every training point but fails on test data (overfit)')


---
## 4 -- Training vs Inference

| | Training | Inference |
|---|---|---|
| **What** | Model LEARNS from data | Model PREDICTS on new data |
| **When** | Before deployment (hours-weeks) | After deployment (milliseconds) |
| **GPU needs** | Massive (multiple GPUs for days) | Moderate (1 GPU per request) |
| **Frequency** | Once (then retrain periodically) | Thousands of times per second |
| **Optimize for** | Quality (lowest loss) | Speed (lowest latency) |

### The Curriculum Connection

```
Training (NB09)          Optimization (NB09)       Serving (NB11-12)
QLoRA fine-tuning   -->  KD, Pruning, Quantize --> BentoML, vLLM, FastAPI
(make model good)        (make model small/fast)   (serve to users)
```

### What Are Embeddings?

An embedding is a **dense vector representation** that captures meaning:

```
"king"   -> [0.8, -0.2, 0.5, 0.1, ...] (384 numbers)
"queen"  -> [0.7, -0.3, 0.6, 0.1, ...] (similar direction!)
"banana" -> [-0.1, 0.9, -0.3, 0.4, ...] (very different direction)
```

**Why they matter:** Vector search (NB18 RAG), semantic caching (NB20), and feature stores (NB07) ALL use embeddings to find similar things.


In [ ]:
# Cell 4 -- Embeddings: visualize semantic similarity

from sklearn.decomposition import PCA

# Simulate word embeddings (in production: model.encode(words))
words = ['king', 'queen', 'prince', 'princess', 'man', 'woman',
         'dog', 'cat', 'puppy', 'kitten',
         'python', 'javascript', 'java', 'coding']

# Create fake but realistic embeddings (royalty cluster, animals cluster, code cluster)
np.random.seed(42)
embeddings = []
for w in words:
    if w in ['king','queen','prince','princess']: base = np.array([2, 3])
    elif w in ['man','woman']: base = np.array([1.5, 2])
    elif w in ['dog','cat','puppy','kitten']: base = np.array([-2, 1])
    else: base = np.array([0, -3])
    embeddings.append(base + np.random.normal(0, 0.3, 2))

embeddings = np.array(embeddings)

plt.figure(figsize=(10, 7))
colors = ['red']*4 + ['orange']*2 + ['blue']*4 + ['green']*4
plt.scatter(embeddings[:, 0], embeddings[:, 1], c=colors, s=100, zorder=3)
for i, w in enumerate(words):
    plt.annotate(w, (embeddings[i, 0]+0.1, embeddings[i, 1]+0.1), fontsize=11)
plt.title('Embeddings: Similar Words Cluster Together', fontsize=14, fontweight='bold')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.grid(alpha=0.3)
plt.savefig('embeddings_visualization.png', dpi=100)
plt.show()
print('Notice: royalty words cluster together, animal words cluster together.')
print('This is how vector search works in RAG (NB18) and semantic caching (NB20).')
print('Similar meaning = similar vector = found by cosine similarity.')


---
## 5 -- Fine-Tuning vs Prompting vs Training From Scratch

| Approach | Data Needed | Cost | Time | When to Use |
|----------|:-:|:-:|:-:|---|
| **Prompting** | 0 examples | $0 | Minutes | General tasks, prototyping |
| **Few-shot prompting** | 3-10 examples | $0 | Minutes | Simple classification/extraction |
| **Fine-tuning (LoRA)** | 1K-10K examples | $10-100 | Hours | Domain-specific behavior |
| **Full fine-tuning** | 10K-100K+ examples | $1K-10K | Days | Maximum quality |
| **Train from scratch** | Trillions of tokens | $1M+ | Months | Only if you're OpenAI/Google |

**Key insight:** Most AI engineers will NEVER train from scratch. You fine-tune existing models or prompt them. That's what NB09 teaches (LoRA/QLoRA fine-tuning).

### What Are Foundation Models?

A **foundation model** (GPT-4, Llama 3, Claude) is pre-trained on massive data. You build on TOP of it:

```
Foundation model (70B params, trained on internet) -- OpenAI/Meta built this
       |
Fine-tune with YOUR 5,000 examples (QLoRA, NB09)  -- YOU do this
       |
Deploy to production (vLLM, BentoML, NB12)          -- YOU do this
```


---
## Summary

| Concept | What You Learned | Where It Appears Next |
|---------|-----------------|----------------------|
| Model = f(input, weights) | ML learns rules from data | Every notebook |
| Training loop | Forward pass, loss, backward, update | NB09 (QLoRA) |
| Loss function | Measures how wrong the model is | NB09, NB14 |
| Learning rate | Controls update size | NB09 (Optuna tunes this) |
| Overfitting | Memorization vs generalization | NB09, NB10 (testing) |
| Training vs inference | Learning vs predicting | NB09 vs NB12 |
| Embeddings | Vector representation of meaning | NB07, 18, 20 |
| Fine-tuning | Adapt pre-trained model to your task | NB09 |

**Next -->** `09_model_development.ipynb` -- QLoRA Fine-Tuning, Knowledge Distillation & Optuna